# Assignment 8: Clustering

## Instructions

* Complete the assignment as outlined below.
* Restart your kernel and rerun your cells before submission.
* Submit your completed notebook (.ipynb).

## Dataset Information

The dataset contains the results of a chemical analysis of wines grown in the same region of Italy but derived from three different cultivars. The analysis measured the quantities of 7 chemical constituents present in each of the three wine types.

The attributes are:

* `Alcohol`
* `Total phenols`
* `Flavanoids`
* `Color intensity`
* `Hue`
* `OD280/OD315 of diluted wines`
* `Proline`
    
Your tasks in this assignment are as follows:

* Perform clustering using K-means and agglomerative clustering, applying only scaling as preprocessing. Compare the outputs of the two methods.
* Apply UMAP to two dimensions and perform clustering again using K-means.
* Interpret the clusters by calculating the mean and standard deviation of each variable within each cluster and describing the characteristics of the resulting clusters.

In [ ]:
# Install yellowbricks and umap if you don't already have it or are in colab
!pip install yellowbrick
!pip install umap-learn[plot]

In [ ]:
# Imports
import numpy as np
import pandas as pd
import umap
import umap.plot

# Clustering
from sklearn.cluster import AgglomerativeClustering, KMeans, SpectralClustering
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import silhouette_samples, silhouette_score
from yellowbrick.cluster import KElbowVisualizer, SilhouetteVisualizer
from yellowbrick.cluster.elbow import kelbow_visualizer

# Plotting
import matplotlib.cm as cm
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
%config InlineBackend.figure_format = 'retina'
%matplotlib inline

In [ ]:
!gdown https://drive.google.com/uc?id=1rxZSgo6UxHD-RRHXqPJ0gzmF6cUWsdqo

## Question 1:

1. Load the dataset wine.csv and display the first 5 rows.
2. Check if there are any null values in the dataset.
3. Display the class distribution of `wine_type`.

In [ ]:
df = pd.read_csv('wine.csv')
print(df.head())

In [ ]:
print('nulls:', df.isnull().sum().sum())

In [ ]:
print(df['wine_type'].value_counts())

## Question 2:

1. For the clustering analyses, create a new dataset `X` by removing the variable wine_type from the dataset df. Display the first five rows of `X`.
2. Standardize all variables in `X`. Display the first five rows of the resulting dataset.

In [ ]:
X = df.drop('wine_type', axis=1)
print(X.head())

In [ ]:
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
print(X_scaled.head())

## Question 3:

Fit a K-means model to the scaled data from Q2 by setting the number of clusters equal to the number of classes in `wine_type`. Then, visualize the resulting clusters using a scatterplot matrix (sns.pairplot).

In [ ]:
n_classes = df['wine_type'].nunique()
km3 = KMeans(n_clusters=n_classes, random_state=42, n_init='auto')
labels3 = km3.fit_predict(X_scaled)
plot_df = X_scaled.copy()
plot_df['cluster'] = labels3.astype(str)
sns.pairplot(plot_df, hue='cluster', corner=True)
plt.suptitle('k-means (k = number of wine classes)', y=1.02)
plt.show()

## Question 4:
1. Use the K-means method to perform clustering on the scaled data from Q2. Determine the optimal number of clusters using the KelbowVisualizer function with k=(2,12).
2. What is the optimal number of clusters using the elbow method?

In [ ]:
elbow = KElbowVisualizer(KMeans(random_state=42, n_init='auto'), k=(2, 12), metric='distortion')
elbow.fit(X_scaled)
elbow.show()
optimal_k = int(elbow.elbow_value_) if elbow.elbow_value_ is not None else 3
print('optimal k (elbow):', optimal_k)

What is the optimal number of clusters according to the elbow method?

see the kelbow plot above; `optimal_k` is printed in the previous cell (typically 3 for this wine data).

## Question 5:

Fit a K-means model on the scaled data using the optimal number of clusters identified in Q4. Then, visualize the resulting clusters using a scatterplot matrix (`sns.pairplot`).

In [ ]:
km_opt = KMeans(n_clusters=optimal_k, random_state=42, n_init='auto')
labels_opt = km_opt.fit_predict(X_scaled)
plot_df2 = X_scaled.copy()
plot_df2['cluster'] = labels_opt.astype(str)
sns.pairplot(plot_df2, hue='cluster', corner=True)
plt.suptitle('k-means (optimal k from elbow)', y=1.02)
plt.show()

## Question 6:

Let's test an agglomerative clustering now with `metric='euclidean'` and `linkage='average'` on the scaled data using the optimal number of clusters identified in Q4. Then, visualize the resulting clusters using a scatterplot matrix (sns.pairplot).


In [ ]:
agg = AgglomerativeClustering(n_clusters=optimal_k, metric='euclidean', linkage='average')
labels_agg = agg.fit_predict(X_scaled)
plot_df3 = X_scaled.copy()
plot_df3['cluster'] = labels_agg.astype(str)
sns.pairplot(plot_df3, hue='cluster', corner=True)
plt.suptitle('agglomerative (euclidean, average)', y=1.02)
plt.show()

## Question 7:

Compare the outputs of K-means in Q5 and agglomerative clustering in Q6.

k-means and agglomerative often agree on coarse structure but differ on border points; average linkage merges clusters differently than k-means centroids, so pairplot hues can differ slightly for the same k.

## Question 8:

1. Apply **UMAP** to the scaled data. Set:
- number of components to 2
- use 30 nearest neighbors
- use cosine metric
- number of training epochs to 1000
- effective minimum distance between embedded points to 0.3
- effective scale of embedded points to 1
- avoids excessive memory use
- do not use a random seed to allow parallel processing.

2. Create a plot, showing 2D projection of our data using UMAP. Remember to add labels and title.

In [ ]:
reducer = umap.UMAP(
    n_components=2, n_neighbors=30, metric='cosine', n_epochs=1000,
    min_dist=0.3, spread=1.0, low_memory=True, random_state=None
)
X_umap = reducer.fit_transform(X_scaled)

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(X_umap[:, 0], X_umap[:, 1], alpha=0.7, s=30)
plt.xlabel('umap 1')
plt.ylabel('umap 2')
plt.title('umap 2d projection')
plt.show()

## Question 9:

Use the results from Q8. Perform K-means clustering again using the optimal number of clusters identified in Q4. Then, visualize the resulting clusters using a scatterplot matrix (sns.pairplot).

In [ ]:
km_umap = KMeans(n_clusters=optimal_k, random_state=42, n_init='auto')
labels_umap = km_umap.fit_predict(X_umap)
umap_df = pd.DataFrame(X_umap, columns=['u1', 'u2'])
umap_df['cluster'] = labels_umap.astype(str)
sns.pairplot(umap_df, hue='cluster', corner=True)
plt.suptitle('k-means on umap embedding', y=1.02)
plt.show()

## Question 10:

Interpret the clusters from Q9 by calculating the mean and standard deviation of each variable within each cluster and describing the characteristics of the resulting clusters.

In [ ]:
X_orig_labeled = X.copy()
X_orig_labeled['cluster'] = labels_umap
means = X_orig_labeled.groupby('cluster').mean()
print('mean by cluster (original features):')
print(means)

In [ ]:
stds = X_orig_labeled.groupby('cluster').std()
print('std by cluster:')
print(stds)

Characteristics of the resulting clusters:

clusters separate mainly on alcohol, color_intensity, proline, and phenols/flavanoids; one cluster tends toward higher alcohol/proline, another toward lower hue and different phenol profiles. std shows within-cluster spread; use the printed mean/std tables above for exact numbers.